# Quandela Cloud 첫 실행 데모

이 노트북은 Quandela Cloud에서 간단한 광자 실험을 한 번 실행해 보는 예제입니다.

1. 토큰을 입력합니다.
2. 빔 스플리터 1개로 된 실험을 만듭니다.
3. 무료 시뮬레이터 `sim:belenos`에서 실행합니다.
4. (선택) 실제 QPU `qpu:belenos`에서 실행합니다.

**실행 전에 필요한 것**

- Cloud 사이트 **Tokens**에서 발급한 토큰 (사용자 매뉴얼 참조)
- QPU에서 실행하려면 관리자가 배정한 QPU 사용 시간과 `qpu:belenos` 플랫폼 배정

In [ ]:
import perceval as pcvl
print("Perceval", pcvl.__version__)

## 1. 토큰 입력

아래 셀을 실행하면 입력란이 나타납니다. Cloud 사이트 **Tokens** 화면에서 **Copy**로 복사한 토큰을 붙여 넣고 Enter를 누르십시오.
토큰은 비밀번호와 같으므로 노트북 안에 직접 적지 않고 이렇게 입력받습니다.

In [ ]:
from getpass import getpass

TOKEN = getpass("토큰을 붙여 넣으세요: ").strip()
print("토큰 길이:", len(TOKEN))

## 2. 실험 만들기

빔 스플리터(BS) 1개에 광자 2개를 하나씩 넣습니다(입력 상태 |1,1⟩).
두 광자가 서로 구별되지 않으면 두 광자가 항상 같은 출구로 나가는 현상(Hong-Ou-Mandel 효과)이 나타나므로, 결과에는 |2,0⟩과 |0,2⟩가 주로 보이고 |1,1⟩은 거의 나오지 않아야 합니다.

In [ ]:
from perceval import BS, BasicState, Experiment

experiment = Experiment(BS())
experiment.with_input(BasicState([1, 1]))
experiment.min_detected_photons_filter(1)

pcvl.pdisplay(BS())

## 3. 실행 함수

플랫폼 이름과 샷 수를 받아 Cloud에 작업을 보내고 결과를 돌려주는 함수입니다.
`job_name`은 Cloud 사이트 **Jobs** 화면에 표시되는 이름입니다.

In [ ]:
from perceval import ExecutionFactory, RemoteComputer, QuandelaCommunicationLayer, acquire


def run_on_cloud(platform, shots, job_name):
    computer = RemoteComputer(QuandelaCommunicationLayer(platform, TOKEN))
    # 원격 플랫폼에서는 max_shots_per_call을 반드시 지정해야 한다
    factory = ExecutionFactory(computer, experiment, max_shots_per_call=shots)
    factory.default_job_name = job_name
    with acquire(computer):
        return factory.sample_count(shots)["results"]


def show(results):
    total = sum(results.values())
    for state, count in sorted(results.items(), key=lambda x: -x[1]):
        print(f"{str(state):>8}  {count:>6}  ({count / total:.1%})")

## 4. 시뮬레이터에서 실행 (무료)

`sim:belenos`는 QPU `qpu:belenos`를 흉내 내는 시뮬레이터이며 사용 시간이 차감되지 않습니다.
작업이 끝날 때까지 수십 초에서 몇 분 정도 걸릴 수 있습니다.

In [ ]:
sim_results = run_on_cloud("sim:belenos", shots=1000, job_name="demo_sim")
show(sim_results)

## 5. 실제 QPU에서 실행 (선택)

**주의:** QPU 실행은 배정받은 QPU 사용 시간을 차감합니다. 대기열 상황에 따라 결과가 나오기까지 오래 걸릴 수 있습니다.

실행하려면 아래 셀의 `RUN_QPU`를 `True`로 바꾼 뒤 실행하십시오.

In [ ]:
RUN_QPU = False

if RUN_QPU:
    qpu_results = run_on_cloud("qpu:belenos", shots=1000, job_name="demo_qpu")
    show(qpu_results)
else:
    print("QPU 실행을 건너뛰었습니다. RUN_QPU = True로 바꾸면 실행됩니다.")

## 6. 결과 확인

Cloud 사이트 왼쪽 메뉴의 **Jobs**에서 `demo_sim`, `demo_qpu` 작업의 상태와 사용 시간을 확인할 수 있습니다.

더 자세한 사용법은 Perceval 문서(https://perceval.quandela.net/docs)의 Remote Computing 튜토리얼을 참고하십시오.